# Final Stage F 51R2 versus 53R2 call-input comparator

This notebook contains: (1) a capture cell to run immediately before the original build cell in Notebook 51R2 or Notebook 53R2; (2) a finalisation cell; and (3) the comparator. Capture both sides in fresh reconstructions, then run the comparator.

The comparator pairs calls by the semantic Boolean mask hash, so historical `M5` versus `M5_REMOVE` naming does not affect pairing. It tests observed input, exact mask, radius, OpenCV method and raw result separately.


## A. Capture setup
Run this in the original notebook immediately before its original reconstruction/build cell. Set `CAPTURE_SIDE` appropriately.


In [ ]:
# ============================================================
# CAPTURE SETUP - RUN IN THE ORIGINAL NOTEBOOK IMMEDIATELY
# BEFORE ITS ORIGINAL CANDIDATE/PAGE-BUILD CELL
#
# Set CAPTURE_SIDE to "51R2" in Notebook 51R2 and "53R2" in
# Notebook 53R2. Use a fresh output/rebuild so cv2.inpaint is
# actually called rather than checkpoints being reused.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import hashlib, inspect, json, platform, sys
import cv2
import numpy as np
import pandas as pd

CAPTURE_SIDE = '51R2'  # CHANGE TO '53R2' IN NOTEBOOK 53R2
PROJECT_ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
CAPTURE_SESSION = 'stage_f_final_call_inputs'

CAPTURE_ROOT = PROJECT_ROOT / CAPTURE_SESSION / CAPTURE_SIDE
CALL_ROOT = CAPTURE_ROOT / 'calls'
CALL_ROOT.mkdir(parents=True, exist_ok=True)

# Clean only this side's previous temporary capture. This does not
# touch candidate pages or historical runs.
for p in sorted(CALL_ROOT.glob('call_*')):
    if p.is_dir():
        import shutil
        shutil.rmtree(p)


def semantic_hash(a):
    a = np.ascontiguousarray(a)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()


def infer_context():
    out = {}
    names = (
        'page', 'page_id', 'page_name', 'region_id', 'region', 'rid',
        'arm', 'restorer', 'rest', 'method', 'regional', 'regional_m5',
        'm5', 'mask', 'observed', 'obs'
    )
    frames = inspect.stack()
    try:
        for fi in frames[2:25]:
            loc = fi.frame.f_locals
            for name in names:
                if name not in loc or name in out:
                    continue
                value = loc[name]
                if isinstance(value, (str, int, float, bool, Path, np.integer, np.floating, np.bool_)):
                    out[name] = str(value)
                elif isinstance(value, np.ndarray):
                    out[name + '_shape'] = list(value.shape)
                    out[name + '_dtype'] = str(value.dtype)
                    out[name + '_sha256'] = semantic_hash(value)
                    if value.ndim == 2:
                        out[name + '_nonzero'] = int((value != 0).sum())
    finally:
        del frames
    return out


if hasattr(cv2, '_stage_f_original_inpaint'):
    _ORIGINAL_INPAINT = cv2._stage_f_original_inpaint
else:
    _ORIGINAL_INPAINT = cv2.inpaint
    cv2._stage_f_original_inpaint = _ORIGINAL_INPAINT

_CAPTURE_ROWS = []
_CALL_NUMBER = 0


def _captured_inpaint(source, mask, radius, flags):
    global _CALL_NUMBER
    _CALL_NUMBER += 1
    source_before = np.ascontiguousarray(np.asarray(source).copy())
    mask_before = np.ascontiguousarray(np.asarray(mask).copy())
    result = _ORIGINAL_INPAINT(source, mask, radius, flags)
    result_array = np.ascontiguousarray(np.asarray(result).copy())
    context = infer_context()

    call_dir = CALL_ROOT / f'call_{_CALL_NUMBER:06d}'
    call_dir.mkdir(parents=True, exist_ok=False)
    np.save(call_dir / 'source.npy', source_before, allow_pickle=False)
    np.save(call_dir / 'mask.npy', mask_before, allow_pickle=False)
    np.save(call_dir / 'result.npy', result_array, allow_pickle=False)

    page = context.get('page') or context.get('page_id') or context.get('page_name')
    region = context.get('region_id') or context.get('region') or context.get('rid')
    method_name = 'INPAINT_TELEA' if int(flags) == int(cv2.INPAINT_TELEA) else str(int(flags))
    row = {
        'side': CAPTURE_SIDE,
        'call_number': _CALL_NUMBER,
        'page': page,
        'region_id': region,
        'radius': float(radius),
        'flags': int(flags),
        'flags_name': method_name,
        'source_shape': list(source_before.shape),
        'source_dtype': str(source_before.dtype),
        'source_sha256': semantic_hash(source_before),
        'mask_shape': list(mask_before.shape),
        'mask_dtype': str(mask_before.dtype),
        'mask_sha256': semantic_hash(mask_before),
        'mask_boolean_sha256': semantic_hash((mask_before > 0).astype(np.uint8)),
        'mask_pixels': int((mask_before > 0).sum()),
        'result_sha256': semantic_hash(result_array),
        'call_directory': str(call_dir),
        'context': context,
    }
    (call_dir / 'metadata.json').write_text(json.dumps(row, indent=2, default=str))
    _CAPTURE_ROWS.append(row)
    print('CAPTURED', CAPTURE_SIDE, _CALL_NUMBER, page, region, method_name, 'mask_pixels=', row['mask_pixels'])
    return result


cv2.inpaint = _captured_inpaint

(CAPTURE_ROOT / 'capture_environment.json').write_text(json.dumps({
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'side': CAPTURE_SIDE,
    'python': sys.version,
    'opencv': cv2.__version__,
    'numpy': np.__version__,
    'platform': platform.platform(),
}, indent=2))

print('Capture installed for:', CAPTURE_SIDE)
print('Capture root:', CAPTURE_ROOT)
print('NOW RUN THE ORIGINAL BUILD CELL UNCHANGED.')


## B. Capture finalisation
Run this after the original reconstruction/build cell completes.


In [ ]:
# ============================================================
# CAPTURE FINALISATION - RUN AFTER THE ORIGINAL BUILD CELL
# ============================================================

if '_ORIGINAL_INPAINT' not in globals():
    raise RuntimeError('Run the capture setup cell first.')

cv2.inpaint = _ORIGINAL_INPAINT

rows = []
for item in _CAPTURE_ROWS:
    flat = {k: v for k, v in item.items() if k != 'context'}
    for k, v in item.get('context', {}).items():
        flat[f'context_{k}'] = v
    rows.append(flat)

capture_df = pd.DataFrame(rows)
if capture_df.empty:
    raise RuntimeError(
        'No cv2.inpaint calls were captured. The original build probably '
        'reused checkpoints. Force a fresh reconstruction and rerun.'
    )

capture_df.to_csv(CAPTURE_ROOT / 'captured_calls.csv', index=False)
(CAPTURE_ROOT / 'capture_complete.json').write_text(json.dumps({
    'side': CAPTURE_SIDE,
    'calls': int(len(capture_df)),
    'telea_calls': int((capture_df.flags_name == 'INPAINT_TELEA').sum()),
    'complete': True,
}, indent=2))

print('Capture complete:', CAPTURE_ROOT)
display(capture_df[['call_number','page','region_id','flags_name','source_shape','mask_pixels']])


## C. Compare completed captures
Run after both `51R2` and `53R2` capture folders contain `capture_complete.json`.


In [ ]:
# ============================================================
# FINAL 51R2-versus-53R2 CALL-INPUT COMPARATOR
#
# Run after both capture directories have been completed.
# It pairs calls primarily by page + exact Boolean-mask identity.
# If region IDs are available on both sides, they are also reported.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, sys
import cv2
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
CAPTURE_BASE = ROOT / 'stage_f_final_call_inputs'
LEFT_ROOT = CAPTURE_BASE / '51R2'
RIGHT_ROOT = CAPTURE_BASE / '53R2'
OUT = CAPTURE_BASE / ('comparison_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
TABLES = OUT / 'tables'
DIFFS = OUT / 'differences'
for p in [OUT, TABLES, DIFFS]:
    p.mkdir(parents=True, exist_ok=False)

FAILING_PAGES = {'Page003','Page004','Page005','Page008','Page036','Page040','Page049'}
EXPECTED_TOTAL_RESIDUAL = 240


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def semantic_hash(a):
    a = np.ascontiguousarray(a)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()

# ============================================================
# MEMORY-SAFE METADATA-FIRST INDEX
# ============================================================

import gc
from collections import defaultdict


def load_metadata_index(root, side):
    """
    Read only metadata.json files.

    No NPY arrays are loaded here.
    """

    complete_path = (
        root /
        'capture_complete.json'
    )

    require(
        complete_path.exists(),
        f'{side} capture is incomplete: {root}'
    )

    rows = []

    for call_dir in sorted(
        (root / 'calls').glob('call_*')
    ):
        metadata_path = (
            call_dir /
            'metadata.json'
        )

        require(
            metadata_path.exists(),
            f'Missing metadata: {metadata_path}'
        )

        metadata = json.loads(
            metadata_path.read_text()
        )

        if (
            metadata.get('flags_name')
            !=
            'INPAINT_TELEA'
        ):
            continue

        rows.append({
            'side':
                side,

            'call_dir':
                call_dir,

            'page':
                metadata.get('page'),

            'region_id':
                metadata.get('region_id'),

            'call_number':
                int(
                    metadata.get(
                        'call_number',
                        -1,
                    )
                ),

            'radius':
                float(
                    metadata['radius']
                ),

            'flags':
                int(
                    metadata['flags']
                ),

            'source_shape':
                tuple(
                    metadata[
                        'source_shape'
                    ]
                ),

            'source_dtype':
                metadata[
                    'source_dtype'
                ],

            'source_sha256':
                metadata[
                    'source_sha256'
                ],

            'mask_shape':
                tuple(
                    metadata[
                        'mask_shape'
                    ]
                ),

            'mask_dtype':
                metadata[
                    'mask_dtype'
                ],

            'mask_sha256':
                metadata[
                    'mask_sha256'
                ],

            'mask_boolean_sha256':
                metadata[
                    'mask_boolean_sha256'
                ],

            'mask_pixels':
                int(
                    metadata[
                        'mask_pixels'
                    ]
                ),

            'result_sha256':
                metadata[
                    'result_sha256'
                ],

            'metadata':
                metadata,
        })

    require(
        rows,
        f'No captured Telea calls for {side}'
    )

    return rows


left_metadata = load_metadata_index(
    LEFT_ROOT,
    '51R2',
)

right_metadata = load_metadata_index(
    RIGHT_ROOT,
    '53R2',
)

print(
    '51R2 Telea metadata rows:',
    len(left_metadata),
)

print(
    '53R2 Telea metadata rows:',
    len(right_metadata),
)


# ------------------------------------------------------------
# Optionally restrict to the known failing pages
# ------------------------------------------------------------

left_pages_available = {
    row['page']
    for row in left_metadata
    if row['page']
}

right_pages_available = {
    row['page']
    for row in right_metadata
    if row['page']
}

page_metadata_available = bool(
    left_pages_available
    and
    right_pages_available
)

if page_metadata_available:

    left_metadata = [
        row
        for row in left_metadata
        if row['page']
        in FAILING_PAGES
    ]

    right_metadata = [
        row
        for row in right_metadata
        if row['page']
        in FAILING_PAGES
    ]

    require(
        left_metadata,
        'No 51R2 calls remain after filtering '
        'to the known failing pages.'
    )

    require(
        right_metadata,
        'No 53R2 calls remain after filtering '
        'to the known failing pages.'
    )

    print(
        'Restricted comparison to:',
        sorted(FAILING_PAGES),
    )

else:

    print(
        'Page metadata is incomplete. '
        'Using semantic-mask pairing across '
        'the complete Telea-call inventory.'
    )


# ------------------------------------------------------------
# Build small metadata index only
# ------------------------------------------------------------

def pairing_key(row):
    """
    Semantic key for an OpenCV Telea call.

    M5 versus M5_REMOVE naming does not matter.
    """

    return (
        row['mask_boolean_sha256'],
        row['radius'],
        row['flags'],
        row['source_shape'],
    )


right_by_key = defaultdict(list)

for row in right_metadata:
    right_by_key[
        pairing_key(row)
    ].append(row)


# ------------------------------------------------------------
# Pair metadata records
# ------------------------------------------------------------

pairs = []
unmatched_left = []
used_right = set()

for left_row in left_metadata:

    key = pairing_key(
        left_row
    )

    candidates = [
        row
        for row in right_by_key.get(
            key,
            [],
        )
        if str(
            row['call_dir']
        )
        not in used_right
    ]

    page = left_row['page']

    if page:
        same_page = [
            row
            for row in candidates
            if row['page'] == page
        ]

        if len(same_page) == 1:
            candidates = same_page

    region_id = left_row[
        'region_id'
    ]

    if (
        region_id
        and len(candidates) > 1
    ):
        same_region = [
            row
            for row in candidates
            if (
                row['region_id']
                ==
                region_id
            )
        ]

        if len(same_region) == 1:
            candidates = same_region

    if len(candidates) == 1:

        right_row = candidates[0]

        used_right.add(
            str(
                right_row[
                    'call_dir'
                ]
            )
        )

        pairs.append((
            left_row,
            right_row,
        ))

    else:

        unmatched_left.append({
            'left_call':
                str(
                    left_row[
                        'call_dir'
                    ]
                ),

            'page':
                page,

            'region_id':
                region_id,

            'pairing_key':
                json.dumps(
                    key,
                    default=str,
                ),

            'candidate_count':
                len(candidates),
        })


unmatched_right = [
    {
        'right_call':
            str(
                row['call_dir']
            ),

        'page':
            row['page'],

        'region_id':
            row['region_id'],

        'pairing_key':
            json.dumps(
                pairing_key(row),
                default=str,
            ),
    }
    for row in right_metadata
    if str(
        row['call_dir']
    )
    not in used_right
]

print(
    'Paired calls:',
    len(pairs),
)

print(
    'Unmatched 51R2:',
    len(unmatched_left),
)

print(
    'Unmatched 53R2:',
    len(unmatched_right),
)


# ------------------------------------------------------------
# Small helper that loads one call only
# ------------------------------------------------------------

def load_call_arrays(
    metadata_row,
    side,
):
    call_dir = metadata_row[
        'call_dir'
    ]

    # mmap_mode='r' prevents NumPy from eagerly copying the
    # complete file into the Python heap.
    source = np.load(
        call_dir / 'source.npy',
        mmap_mode='r',
        allow_pickle=False,
    )

    mask = np.load(
        call_dir / 'mask.npy',
        mmap_mode='r',
        allow_pickle=False,
    )

    result = np.load(
        call_dir / 'result.npy',
        mmap_mode='r',
        allow_pickle=False,
    )

    require(
        semantic_hash(source)
        ==
        metadata_row[
            'source_sha256'
        ],
        f'{side}: source hash failure: '
        f'{call_dir}'
    )

    require(
        semantic_hash(mask)
        ==
        metadata_row[
            'mask_sha256'
        ],
        f'{side}: mask hash failure: '
        f'{call_dir}'
    )

    require(
        semantic_hash(result)
        ==
        metadata_row[
            'result_sha256'
        ],
        f'{side}: result hash failure: '
        f'{call_dir}'
    )

    return source, mask, result


comparison_rows = []
coordinate_rows = []

# ============================================================
# STREAMING PAIR COMPARISON
# ============================================================

for pair_number, (
    left_row,
    right_row,
) in enumerate(
    pairs,
    start=1,
):

    ls = lm_raw = lr = None
    rs = rm_raw = rr = None

    try:

        ls, lm_raw, lr = (
            load_call_arrays(
                left_row,
                '51R2',
            )
        )

        rs, rm_raw, rr = (
            load_call_arrays(
                right_row,
                '53R2',
            )
        )

        lm = lm_raw > 0
        rm = rm_raw > 0

        same_shape = (
            ls.shape
            ==
            rs.shape
            ==
            lm.shape
            ==
            rm.shape
            ==
            lr.shape
            ==
            rr.shape
        )

        if not same_shape:

            comparison_rows.append({
                'page_51':
                    left_row['page'],

                'page_53':
                    right_row['page'],

                'region_51':
                    left_row[
                        'region_id'
                    ],

                'region_53':
                    right_row[
                        'region_id'
                    ],

                'call_51':
                    str(
                        left_row[
                            'call_dir'
                        ]
                    ),

                'call_53':
                    str(
                        right_row[
                            'call_dir'
                        ]
                    ),

                'pairing_method':
                    'METADATA_SEMANTIC_KEY',

                'shape_exact':
                    False,

                'diagnosis':
                    'SHAPE_DIFFERENCE',
            })

            continue

        source_exact = bool(
            np.array_equal(
                ls,
                rs,
            )
        )

        mask_exact = bool(
            np.array_equal(
                lm,
                rm,
            )
        )

        radius_exact = bool(
            left_row['radius']
            ==
            right_row['radius']
        )

        flags_exact = bool(
            left_row['flags']
            ==
            right_row['flags']
        )

        result_exact = bool(
            np.array_equal(
                lr,
                rr,
            )
        )

        if source_exact:
            source_different_pixels = 0
        else:
            source_different_pixels = int(
                np.count_nonzero(
                    ls != rs
                )
            )

        if mask_exact:
            mask_xor_pixels = 0
        else:
            mask_xor_pixels = int(
                np.count_nonzero(
                    lm != rm
                )
            )

        if result_exact:
            result_different_pixels = 0
            result_inside_union = 0
            result_max_abs = 0

        else:
            result_difference = (
                lr != rr
            )

            union_mask = (
                lm | rm
            )

            result_different_pixels = int(
                np.count_nonzero(
                    result_difference
                )
            )

            result_inside_union = int(
                np.count_nonzero(
                    result_difference
                    &
                    union_mask
                )
            )

            result_max_abs = int(
                np.max(
                    np.abs(
                        lr.astype(
                            np.int16
                        )
                        -
                        rr.astype(
                            np.int16
                        )
                    )
                )
            )

        if not source_exact:

            diagnosis = (
                'OBSERVED_INPUT_DIFFERS'
            )

            first_correction = (
                'Make 53R2 load the exact '
                'observed array used by 51R2.'
            )

        elif not mask_exact:

            diagnosis = (
                'REGIONAL_MASK_DIFFERS'
            )

            first_correction = (
                'Make 53R2 construct the '
                'regional M5 exactly as 51R2.'
            )

        elif (
            not radius_exact
            or
            not flags_exact
        ):

            diagnosis = (
                'INPAINT_PARAMETERS_DIFFER'
            )

            first_correction = (
                'Use the same Telea radius '
                'and OpenCV flag.'
            )

        elif not result_exact:

            diagnosis = (
                'IDENTICAL_INPUTS_BUT_'
                'RAW_RESULT_DIFFERS'
            )

            first_correction = (
                'Inputs match. Compare runtime, '
                'array layout and OpenCV build.'
            )

        else:

            diagnosis = 'CALL_EXACT'

            first_correction = (
                'Call is exact. Inspect regional '
                'ordering or post-inpaint composition.'
            )

        comparison_rows.append({
            'pair_number':
                pair_number,

            'page_51':
                left_row['page'],

            'page_53':
                right_row['page'],

            'region_51':
                left_row[
                    'region_id'
                ],

            'region_53':
                right_row[
                    'region_id'
                ],

            'call_51':
                str(
                    left_row[
                        'call_dir'
                    ]
                ),

            'call_53':
                str(
                    right_row[
                        'call_dir'
                    ]
                ),

            'pairing_method':
                'METADATA_SEMANTIC_KEY',

            'shape_exact':
                True,

            'source_exact':
                source_exact,

            'source_different_pixels':
                source_different_pixels,

            'mask_exact':
                mask_exact,

            'mask_xor_pixels':
                mask_xor_pixels,

            'mask_pixels_51':
                int(lm.sum()),

            'mask_pixels_53':
                int(rm.sum()),

            'radius_51':
                left_row['radius'],

            'radius_53':
                right_row['radius'],

            'radius_exact':
                radius_exact,

            'flags_51':
                left_row['flags'],

            'flags_53':
                right_row['flags'],

            'flags_exact':
                flags_exact,

            'result_exact':
                result_exact,

            'result_different_pixels':
                result_different_pixels,

            'result_different_inside_mask_union':
                result_inside_union,

            'result_max_absolute_difference':
                result_max_abs,

            'diagnosis':
                diagnosis,

            'first_correction':
                first_correction,

            'source_sha256_51':
                left_row[
                    'source_sha256'
                ],

            'source_sha256_53':
                right_row[
                    'source_sha256'
                ],

            'mask_boolean_sha256_51':
                left_row[
                    'mask_boolean_sha256'
                ],

            'mask_boolean_sha256_53':
                right_row[
                    'mask_boolean_sha256'
                ],

            'result_sha256_51':
                left_row[
                    'result_sha256'
                ],

            'result_sha256_53':
                right_row[
                    'result_sha256'
                ],
        })

        # Only write large diagnostic images for non-exact pairs.
        if diagnosis != 'CALL_EXACT':

            page_label = (
                left_row['page']
                or
                right_row['page']
                or
                'UNKNOWN_PAGE'
            )

            region_label = (
                left_row['region_id']
                or
                right_row['region_id']
                or
                f'pair_{pair_number:06d}'
            )

            safe_name = (
                f'{page_label}__{region_label}'
                .replace('/', '_')
                .replace('\\', '_')
                .replace(':', '_')
                .replace(' ', '_')
            )

            difference_directory = (
                DIFFS /
                safe_name
            )

            difference_directory.mkdir(
                parents=True,
                exist_ok=True,
            )

            if not source_exact:

                source_absdiff = np.abs(
                    ls.astype(np.int16)
                    -
                    rs.astype(np.int16)
                ).astype(np.uint8)

                cv2.imwrite(
                    str(
                        difference_directory /
                        'source_absolute_difference.png'
                    ),
                    source_absdiff,
                )

                del source_absdiff

            if not mask_exact:

                cv2.imwrite(
                    str(
                        difference_directory /
                        'mask_xor.png'
                    ),
                    (
                        lm != rm
                    ).astype(np.uint8)
                    * 255,
                )

            if not result_exact:

                result_absdiff = np.abs(
                    lr.astype(np.int16)
                    -
                    rr.astype(np.int16)
                ).astype(np.uint8)

                cv2.imwrite(
                    str(
                        difference_directory /
                        'result_absolute_difference.png'
                    ),
                    result_absdiff,
                )

                del result_absdiff

            # Coordinate rows are generated only for non-exact pairs.
            combined_difference = (
                (ls != rs)
                |
                (lm != rm)
                |
                (lr != rr)
            )

            ys, xs = np.where(
                combined_difference
            )

            for y, x in zip(ys, xs):

                coordinate_rows.append({
                    'page':
                        page_label,

                    'region':
                        region_label,

                    'y':
                        int(y),

                    'x':
                        int(x),

                    'source_51':
                        int(ls[y, x]),

                    'source_53':
                        int(rs[y, x]),

                    'mask_51':
                        bool(lm[y, x]),

                    'mask_53':
                        bool(rm[y, x]),

                    'result_51':
                        int(lr[y, x]),

                    'result_53':
                        int(rr[y, x]),
                })

            del combined_difference

        if (
            pair_number % 50 == 0
            or
            pair_number == len(pairs)
        ):

            print(
                f'Compared '
                f'{pair_number:,}/'
                f'{len(pairs):,} pairs'
            )

    finally:

        # Release memmaps and temporary Boolean arrays before
        # reading the next pair.
        del ls, lm_raw, lr
        del rs, rm_raw, rr

        if 'lm' in locals():
            del lm

        if 'rm' in locals():
            del rm

        if 'result_difference' in locals():
            del result_difference

        if 'union_mask' in locals():
            del union_mask

        gc.collect()


comparison_df = pd.DataFrame(comparison_rows)
coordinate_df = pd.DataFrame(coordinate_rows, columns=['page','region','y','x','source_51','source_53','mask_51','mask_53','result_51','result_53'])
unmatched_left_df = pd.DataFrame(unmatched_left)
unmatched_right_df = pd.DataFrame(unmatched_right)

comparison_df.to_csv(TABLES / 'call_input_comparison.csv', index=False)
coordinate_df.to_csv(TABLES / 'call_input_difference_coordinates.csv', index=False)
unmatched_left_df.to_csv(TABLES / 'unmatched_51r2_calls.csv', index=False)
unmatched_right_df.to_csv(TABLES / 'unmatched_53r2_calls.csv', index=False)

counts = comparison_df.diagnosis.value_counts().to_dict() if len(comparison_df) else {}
all_pairs_exact = bool(len(comparison_df) and (comparison_df.diagnosis == 'CALL_EXACT').all())

if counts.get('OBSERVED_INPUT_DIFFERS', 0):
    decisive_diagnosis = 'OBSERVED_INPUT_DIFFERENCE_FOUND'
elif counts.get('REGIONAL_MASK_DIFFERS', 0):
    decisive_diagnosis = 'REGIONAL_MASK_DIFFERENCE_FOUND'
elif counts.get('INPAINT_PARAMETERS_DIFFER', 0):
    decisive_diagnosis = 'INPAINT_PARAMETER_DIFFERENCE_FOUND'
elif counts.get('IDENTICAL_INPUTS_BUT_RAW_RESULT_DIFFERS', 0):
    decisive_diagnosis = 'RUNTIME_OR_MEMORY_LAYOUT_DIFFERENCE'
elif all_pairs_exact and not unmatched_left and not unmatched_right:
    decisive_diagnosis = 'CALLS_EXACT_DIFFERENCE_IS_POST_INPAINT_COMPOSITION'
else:
    decisive_diagnosis = 'PAIRING_INCOMPLETE_REVIEW_UNMATCHED_CALLS'

verdict = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': 'Fresh 51R2 versus 53R2 cv2.inpaint call-input comparison',
    'paired_calls': int(len(comparison_df)),
    'unmatched_51r2_calls': int(len(unmatched_left)),
    'unmatched_53r2_calls': int(len(unmatched_right)),
    'diagnosis_counts': {str(k): int(v) for k,v in counts.items()},
    'decisive_diagnosis': decisive_diagnosis,
    'expected_historical_stage_f_residual_pixels': EXPECTED_TOTAL_RESIDUAL,
    'stage_f_pass_declared': False,
    'software': {
        'python': sys.version,
        'opencv': cv2.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'platform': platform.platform(),
    },
}
(OUT / 'call_input_comparator_verdict.json').write_text(json.dumps(verdict, indent=2))

print(json.dumps(verdict, indent=2))
print('\nComparison table:')
display(comparison_df.sort_values(['diagnosis','page_51','region_51']))
print('\nOutputs:', OUT)
